In [ ]:
# --- STEP 1: Install dependencies ---
!pip install fastapi uvicorn pyngrok nest_asyncio pydantic sentence-transformers chromadb scikit-learn pyyaml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.5 MB/s et

In [ ]:
# --- STEP 2: Create config.yaml ---
config = """
model:
  name: all-MiniLM-L6-v2
  device: cpu

processing:
  max_logs_in_memory: 200

clustering:
  eps: 0.8
  min_samples: 3
"""

with open("config.yaml", "w") as f:
    f.write(config)


In [ ]:
# --- STEP 3: Paste your FastAPI app code ---
# (Exactly as you provided, but remove the "if __name__ == '__main__'" block)
# Keep everything above that point exactly the same.

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import asyncio
import time
import yaml
from datetime import datetime

from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.cluster import DBSCAN
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Pydantic models
class LogEntry(BaseModel):
    message: str
    timestamp: str
    source: str

class LogBatch(BaseModel):
    logs: List[LogEntry]

class AnomalyResponse(BaseModel):
    anomaly: bool
    confidence: float
    reason: Optional[str] = None
    cluster_label: Optional[int] = None

class MLService:
    def __init__(self, config_path: str = "config.yaml"):
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)

        self.model = SentenceTransformer(
            self.config['model']['name'],
            device=self.config['model']['device']
        )

        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = self.chroma_client.get_or_create_collection(
            name="log_embeddings",
            metadata={"description": "Log embeddings for anomaly detection"}
        )

        self.processed_count = 0
        logger.info("ML Service initialized")

    async def process_log_batch(self, batch: LogBatch) -> dict:
        try:
            embeddings, documents, metadatas, ids = [], [], [], []
            for log_entry in batch.logs:
                embedding = self.model.encode([log_entry.message])[0].tolist()
                embeddings.append(embedding)
                documents.append(log_entry.message)
                metadatas.append({
                    "timestamp": log_entry.timestamp,
                    "source": log_entry.source,
                    "processed_at": time.time()
                })
                ids.append(f"log_{int(time.time() * 1000)}_{self.processed_count}")
                self.processed_count += 1

            if embeddings:
                self.collection.add(
                    embeddings=embeddings,
                    documents=documents,
                    metadatas=metadatas,
                    ids=ids
                )

            anomalies = await self.detect_anomalies()
            return {
                "status": "processed",
                "processed": len(batch.logs),
                "anomalies_detected": len(anomalies),
                "total_processed": self.processed_count
            }

        except Exception as e:
            logger.error(f"Error processing batch: {e}")
            raise HTTPException(status_code=500, detail=str(e))

    async def detect_anomalies(self) -> List[AnomalyResponse]:
        try:
            results = self.collection.get(
                limit=self.config['processing']['max_logs_in_memory'],
                include=['embeddings', 'documents', 'metadatas']
            )
            if len(results['embeddings']) < self.config['clustering']['min_samples']:
                return []

            embeddings = np.array(results['embeddings'])
            clustering = DBSCAN(
                eps=self.config['clustering']['eps'],
                min_samples=self.config['clustering']['min_samples']
            ).fit(embeddings)

            anomalies = []
            for idx, label in enumerate(clustering.labels_):
                if label == -1:
                    anomaly = AnomalyResponse(
                        anomaly=True,
                        confidence=0.95,
                        reason="Semantic outlier detected",
                        cluster_label=int(label)
                    )
                    anomalies.append(anomaly)
                    logger.warning(f"Anomaly detected: {results['documents'][idx]}")

            return anomalies

        except Exception as e:
            logger.error(f"Error in anomaly detection: {e}")
            return []

    async def analyze_single_log(self, log_message: str) -> AnomalyResponse:
        try:
            embedding = self.model.encode([log_message])[0].reshape(1, -1)
            results = self.collection.get(limit=100, include=['embeddings'])

            if len(results['embeddings']) > 0:
                existing_embeddings = np.array(results['embeddings'])
                all_embeddings = np.vstack([existing_embeddings, embedding])
                clustering = DBSCAN(
                    eps=self.config['clustering']['eps'],
                    min_samples=self.config['clustering']['min_samples']
                ).fit(all_embeddings)

                if clustering.labels_[-1] == -1:
                    return AnomalyResponse(
                        anomaly=True,
                        confidence=0.90,
                        reason="Semantic outlier in current context",
                        cluster_label=int(clustering.labels_[-1])
                    )

            return AnomalyResponse(
                anomaly=False,
                confidence=0.85,
                reason="Pattern matches known log clusters"
            )

        except Exception as e:
            logger.error(f"Error in single log analysis: {e}")
            return AnomalyResponse(
                anomaly=False,
                confidence=0.0,
                reason=f"Analysis error: {str(e)}"
            )

# FastAPI app
app = FastAPI(title="Log Anomaly Detection API", version="1.0.0")
ml_service = MLService()

@app.post("/ingest")
async def ingest_logs(batch: LogBatch):
    return await ml_service.process_log_batch(batch)

@app.post("/analyze")
async def analyze_log(log_entry: LogEntry):
    return await ml_service.analyze_single_log(log_entry.message)

@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "model_loaded": True,
        "processed_count": ml_service.processed_count,
        "timestamp": datetime.now().isoformat()
    }

@app.get("/metrics")
async def get_metrics():
    return {
        "total_processed": ml_service.processed_count,
        "vector_store_size": ml_service.collection.count(),
        "status": "operational"
    }


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!ngrok authtoken 34YHdlT7DJfQm2dVADOv2cV5lgz_5nPfZgq2NZRjdH2BRWQMZ

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# Allow nested event loops
nest_asyncio.apply()

# Create a public URL for your FastAPI app
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

# # Launch FastAPI inside the notebook event loop
# uvicorn.run(app, host="0.0.0.0", port=8000)


from uvicorn import Config, Server

config = Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = Server(config=config)

# Run manually within the existing loop
await server.serve()


Public URL: NgrokTunnel: "https://combinedly-toponymic-kennith.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [314]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET /embed HTTP/1.1" 404 Not Found
INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET /analyze/hellosdflasdf HTTP/1.1" 404 Not Found
INFO:     104.28.255.117:0 - "GET /health HTTP/1.1" 200 OK


INFO:     104.28.255.117:0 - "POST /ingest HTTP/1.1" 200 OK
INFO:     104.28.255.117:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     104.28.255.117:0 - "GET /metrics HTTP/1.1" 200 OK
INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2a09:bac5:3da2:eaa::176:78:0 - "GET /openapi.json HTTP/1.1" 200 OK


In [ ]:
import requests

# Paste your ngrok public URL here
base_url = "https://combinedly-toponymic-kennith.ngrok-free.dev/"

# 1️⃣ Health check
print("Health check:")
print(requests.get(f"{base_url}/health").json())

# 2️⃣ Ingest a batch of logs
print("\nIngest logs:")
logs = {
    "logs": [
        {"message": "User login successful", "timestamp": "2025-10-25T12:00:00Z", "source": "auth_service"},
        {"message": "Database timeout on request /api/orders", "timestamp": "2025-10-25T12:01:00Z", "source": "db_service"},
        {"message": "Unhandled exception in payment processor", "timestamp": "2025-10-25T12:02:00Z", "source": "payment_service"}
    ]
}
print(requests.post(f"{base_url}/ingest", json=logs).json())

# 3️⃣ Analyze a single log
print("\nAnalyze single log:")
log_entry = {
    "message": "Critical memory leak detected in worker 3",
    "timestamp": "2025-10-25T12:10:00Z",
    "source": "worker_service"
}
print(requests.post(f"{base_url}/analyze", json=log_entry).json())

# 4️⃣ Get metrics
print("\nMetrics:")
print(requests.get(f"{base_url}/metrics").json())
